In [1]:
# ============================================================
# Phase 2 — HE-Friendly UNet on ACDC
# poly + InstanceNorm + pretrained, 400 epochs
# ============================================================

In [2]:
# CELL 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# CELL 2 — Clone repo
import os
if not os.path.exists('/content/acdc-he-segmentation'):
    os.system('git clone https://github.com/masiirene/acdc-he-segmentation.git /content/acdc-he-segmentation')
else:
    print('Repo already cloned')

os.chdir('/content/acdc-he-segmentation')
os.system('pip install nibabel -q')

import sys
sys.path.insert(0, '/content/acdc-he-segmentation')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# CELL 3 — Copy data to RAM
os.makedirs('/content/acdc_data', exist_ok=True)
os.makedirs('data', exist_ok=True)

if not os.path.exists('/content/acdc_data/training'):
    os.system('cp -r /content/drive/MyDrive/tesi/tesi_acdc/training /content/acdc_data/')
    print('Training data copied')
else:
    print('Training data already exists')

if not os.path.exists('/content/acdc_data/splits_final.json'):
    os.system('cp /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_preprocessed/Dataset027_ACDC/splits_final.json /content/acdc_data/')
    print('splits_final.json copied')

if not os.path.exists('data/baseline_weights.pth'):
    os.system('cp /content/drive/MyDrive/tesi/baseline_weights.pth data/')
    print('baseline_weights.pth copied')

print('Data ready!')
print(f'Patients: {len(os.listdir("/content/acdc_data/training"))}')

Training data copied
splits_final.json copied
baseline_weights.pth copied
Data ready!
Patients: 102


In [ ]:
# CELL 4 — Launch training
!PYTHONPATH=/content/acdc-he-segmentation python training/train.py \
    --act poly \
    --norm instance \
    --pretrained data/baseline_weights.pth \
    --batch_size 16 \
    --lr 1e-4 \
    --epochs 400 \
    --early_stop 400 \
    --data_dir /content/acdc_data/training \
    --out_dir results/poly_instance_400ep

Device: cuda
Train cases: 160, Val cases: 40
Train slices: 1534, Val slices: 368
Model: act=poly, norm=instance, params=20,617,286
Loaded 44 conv layers, skipped 2
Epoch   1 | loss 1.9408 | val_loss 1.6918 | RV 0.516 MYO 0.338 LV 0.768 | mean 0.540 | lr 1.00e-04
  → saved best model (mean dice 0.540)
Epoch   2 | loss 1.5220 | val_loss 1.4217 | RV 0.606 MYO 0.502 LV 0.840 | mean 0.649 | lr 1.00e-04
  → saved best model (mean dice 0.649)
Epoch   3 | loss 1.2698 | val_loss 1.2023 | RV 0.700 MYO 0.628 LV 0.857 | mean 0.728 | lr 1.00e-04
  → saved best model (mean dice 0.728)
Epoch   4 | loss 1.0541 | val_loss 0.9957 | RV 0.738 MYO 0.706 LV 0.878 | mean 0.774 | lr 1.00e-04
  → saved best model (mean dice 0.774)
Epoch   5 | loss 0.8702 | val_loss 0.8329 | RV 0.761 MYO 0.757 LV 0.883 | mean 0.800 | lr 1.00e-04
  → saved best model (mean dice 0.800)
Epoch   6 | loss 0.7147 | val_loss 0.7003 | RV 0.738 MYO 0.742 LV 0.879 | mean 0.786 | lr 1.00e-04
Epoch   7 | loss 0.5731 | val_loss 0.5847 | RV 

In [ ]:
# CELL 5 — Save to Drive (run this periodically in a separate tab)
import shutil, glob
os.makedirs('/content/drive/MyDrive/tesi/phase2_results', exist_ok=True)
results = glob.glob('results/poly_instance_400ep/**/*.pth', recursive=True)
results += glob.glob('results/poly_instance_400ep/**/*.json', recursive=True)
for f in results:
    dst = f'/content/drive/MyDrive/tesi/phase2_results/{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved: {os.path.basename(f)}')
print('Done!')